# Expanded routing criteria and batched GPT-5.6 Sol annotation

This is dataset preparation, not an LLM-versus-Clef evaluation. The original six-route experiment stays unchanged. The new coding track has 14 proposed subagent roles; the separate tool/API track has seven. These are experiment contracts, not a claim that these agents are already installed in GitHub Copilot.

This notebook shows the input boundary, expected JSON format and saved results for the updated criteria. Up to ten independent rows share one request; a second blind request changes criterion/input order without seeing the first vote. Agreement across both blind passes is AI-assisted silver labeling, not human-reviewed gold. Disagreements require adjudication.

The seven complete rows from the earlier one-row-per-request serial run are preserved. No training, downstream evaluation or benchmark-tool execution occurs here. All remote actions default to disabled.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'routing_lab').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the project workspace.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from routing_lab.annotation import load_plan, jsonl, _completed
from routing_lab.annotation_judge import load_judge_config
from routing_lab.batch_format import Batch, build_batches, batch_payload, batch_prompt, batch_schema, batch_characters

PLAN = ROOT / 'data' / 'public-routing' / 'public-annotation-20260930-v2'
CONFIG = ROOT / 'azure' / 'annotation-judge-batched.json'
SERIAL_RUN = ROOT / 'outputs' / 'public-annotation-gpt56-sol-20261001'
OBSERVED_RUN = ROOT / 'outputs' / 'public-annotation-batched-gpt56-sol-20261001'
manifest, items, catalogs = load_plan(PLAN)
ready = [row for row in items if row.eligibility == 'ready']
config = load_judge_config(CONFIG)
print('Judge:', config.model_dump())
print('Tracks:', {name: len(contracts) for name, contracts in catalogs.items()})
print('Eligibility:', manifest['eligibility_counts'])

## Routing criteria

Both the LLM router and Clef see the same proposed-owner descriptions below. Coding requests are not automatically tool-executable: the same request can have a coding owner but no matching capability in a restricted tool roster.

In [ ]:
for taxonomy, contracts in catalogs.items():
    print('\n', taxonomy, ':', len(contracts), 'routes')
    for route, contract in contracts.items():
        print(f'\n{route}: {contract.description}')

## What is sent

The examples below are already development-exposed. Only their query/context enters the task payload, under opaque keys. The routing criteria appear once in the system prompt. Native IDs, dataset categories, old labels and native solutions are not added.

Real batches never mix taxonomies, native splits or development-exposure status. The second pass reverses entry order but preserves each key.

In [ ]:
first = next(row for row in ready if row.development_exposed)
preview_rows = [row for row in ready if (row.taxonomy_id, row.native_split, row.development_exposed) == (first.taxonomy_id, first.native_split, first.development_exposed)][:2]
preview_batch = Batch(tuple(preview_rows))
print('Task payload, first pass:')
print(json.dumps(batch_payload(preview_batch, 0), indent=2, ensure_ascii=False))
print('\nSecond-pass key order:', list(batch_payload(preview_batch, 1)['items']))
print('Allowed routes:', list(catalogs[preview_batch.taxonomy_id]))
print('Shared system-prompt characters:', len(batch_prompt(catalogs[preview_batch.taxonomy_id], 0)))

## Expected JSON response

The schema requires exactly one annotation for every supplied opaque key. Each annotation contains a verdict, primary route, nullable alternative, brief rationale, **one evidence object**, and nullable missing-information explanation. The evidence quote must be non-empty but is kept as unverified audit context, not re-checked against that row's input: row attribution is already guaranteed by the opaque key mapping itself (see the Approved bounded retries / evidence section in docs/annotation.md for why the stricter check was removed). The client still rejects missing keys, extra keys and invalid label/abstention combinations.

Using one evidence object prevents the earlier six-quotes failure. The nullable fixed slots avoid unsupported Azure array-count constraints. Returned object order does not control row matching.

In [ ]:
print(json.dumps(batch_schema(preview_batch), indent=2))

In [ ]:
serial_events = jsonl(SERIAL_RUN / 'events.jsonl')
serial_votes = _completed(serial_events, ready)
reused_ids = {row.id for row in ready if all((row.id, index) in serial_votes for index in (0, 1))}
remaining = [row for row in ready if row.id not in reused_ids]
batches = build_batches(remaining, catalogs, batch_size=100)
print('Completed serial rows to preserve:', len(reused_ids))
print('Remaining rows:', len(remaining), 'Batches:', len(batches))
print('New successful requests required:', 2 * len(batches))
print('Largest full request, characters:', max(batch_characters(batch, catalogs[batch.taxonomy_id]) for batch in batches))
print('The runner independently validates the serial audit and charges all original attempts, including its failed request.')

## Optional budgeted run

Leave this disabled while the existing worker is running. The separate notebook output path prevents accidental reuse of its directory. Set a positive cumulative request cap and explicitly enable the action before sending anything. Imported attempts count against the cap; 17 permits the original 15 serial attempts plus one new batch's two calls. The full current recovery plan at batch_size=100 needs about 267 cumulative attempts if no additional request fails (versus 1,251 at the earlier fixed batch_size=10).

For resume, use the same configuration and output with RESUME=True; successful passes are not repeated. The base coordinator stops on failure. An explicitly enabled retry worker allows one retry per rejected batch pass, with a separate cumulative retry allowance. It never weakens evidence checks or retries service/authentication/integrity failures. Authentication is the Azure CLI identity through Microsoft Entra, not a static key.

In [ ]:
RUN_ANNOTATION = False
MAX_REQUESTS = 0
RESUME = False
ALLOW_BOUNDED_RETRIES = False
MAX_RETRY_REQUESTS = 0
RUN_DIR = ROOT / 'outputs' / 'public-annotation-batched-gpt56-sol-notebook-001'
if RUN_ANNOTATION:
    if MAX_REQUESTS <= 0:
        raise ValueError('Approve a positive cumulative request cap first.')
    if ALLOW_BOUNDED_RETRIES:
        if not RESUME or MAX_RETRY_REQUESTS <= 0:
            raise ValueError('Bounded retries require an existing run and an explicit positive retry allowance.')
        command = [sys.executable, '-m', 'routing_lab.batch_annotation', 'retry', '--plan', str(PLAN), '--judge-config', str(CONFIG), '--output', str(RUN_DIR), '--max-requests', str(MAX_REQUESTS), '--max-retry-requests', str(MAX_RETRY_REQUESTS), '--allow-remote']
    else:
        command = [sys.executable, '-m', 'routing_lab.batch_annotation', 'run', '--plan', str(PLAN), '--judge-config', str(CONFIG), '--output', str(RUN_DIR), '--batch-size', '100', '--max-requests', str(MAX_REQUESTS), '--allow-remote']
        command += ['--resume'] if RESUME else ['--import-serial-run', str(SERIAL_RUN)]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Remote annotation disabled. This cell sent no requests.')

In [ ]:
report_path = OBSERVED_RUN / 'manifest.json'
if report_path.exists():
    saved_report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps(saved_report, indent=2, ensure_ascii=False))
else:
    print('No saved batched run yet. The cells above only prepared and inspected inputs.')

## Actual saved request and responses

This cell reads the first completed batch, not a fabricated example, and shows one already development-exposed row from it. The displayed input/output are excerpts of the ten-row request and response. Both saved blind assessments are shown when available. No additional model call is made.

In [ ]:
from routing_lab.batch_format import validate_batch

events_path = OBSERVED_RUN / 'batch_events.jsonl'
if events_path.exists():
    completed_events = [event for event in jsonl(events_path) if event['event'] == 'completed']
    if completed_events:
        first_event = completed_events[0]
        identity = json.loads((OBSERVED_RUN / 'identity.json').read_text(encoding='utf-8'))
        descriptor = next(batch for batch in identity['batches'] if batch['id'] == first_event['batch_id'])
        by_id = {row.id: row for row in items}
        actual_batch = Batch(tuple(by_id[row_id] for row_id in descriptor['ids']))
        exposed_slots = [(key, row) for key, row in actual_batch.slots.items() if row.development_exposed]
        if not exposed_slots:
            raise ValueError('This batch has no already development-exposed example to display.')
        slot, actual_row = exposed_slots[0]
        print('One actual input entry; full batch size:', len(actual_batch.items))
        print(json.dumps({'items': {slot: actual_row.model_input()}}, indent=2, ensure_ascii=False))
        for event in completed_events:
            if event['batch_id'] == actual_batch.id:
                validate_batch(event['payload'], actual_batch)
                print('\nActual response, blind pass', event['pass_index'] + 1, 'model:', event['model'])
                print(json.dumps({'annotations': {slot: event['payload']['annotations'][slot]}}, indent=2, ensure_ascii=False))
                print('Whole-batch usage:', event['input_tokens'], 'input tokens;', event['output_tokens'], 'output tokens')
    else:
        print('No completed batch response is saved yet.')
else:
    print('No batch journal exists yet.')

In [ ]:
from collections import Counter

policy_path = OBSERVED_RUN / 'retry_policy.json'
if policy_path.exists():
    policy = json.loads(policy_path.read_text(encoding='utf-8'))
    attempts = Counter((event['batch_id'], event['pass_index']) for event in jsonl(OBSERVED_RUN / 'batch_events.jsonl') if event['event'] == 'started')
    print('Retry policy:', json.dumps(policy, indent=2))
    print('Retry requests used:', sum(count - 1 for count in attempts.values()))
    print('Maximum attempts observed for one batch pass:', max(attempts.values(), default=0))
    print('Original failed responses stay in the journal; no evidence is repaired or accepted silently.')
else:
    print('No bounded-retry policy has been registered for this run.')

## Before evaluation or training

Export only validated two-pass agreement. Keep disagreements for adjudication and preserve provenance, native splits and known development exposure. Review a representative sample, inspect class coverage, and freeze group-aware training/evaluation boundaries before using labels. AI agreement is not evidence of human correctness, and the annotating GPT deployment must not be presented as an independent evaluation judge of itself.

The 1,055 context-blocked rows and ten ToolBench examples remain excluded. The live Clef endpoint still needs deliberate deployment/verification for the expanded catalogs. See [annotation](../docs/annotation.md) for the run and export commands.